# LSTM & GRU Fundamentals

Gated recurrent cells derived from scratch with full BPTT, validated against PyTorch,
and compared to vanilla RNNs on gradient flow and long-range dependency tasks.

We answer four concrete questions with numbers:
1. How do LSTM and GRU gates modify the vanilla RNN update?
2. Do from-scratch BPTT gradients match `nn.LSTM` and `nn.GRU`?
3. Does an LSTM preserve gradient magnitude better than a vanilla RNN through time?
4. On delayed recall (Topic 10's failure case), do gated cells learn where vanilla RNNs fail?

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(0)
plt.rcParams['figure.dpi'] = 100

## 1. From-Scratch LSTM & GRU

**LSTM** adds a cell state $c_t$ (long-term memory) and three gates:

$$i_t = \sigma(x_t W_{xi} + h_{t-1} W_{hi} + b_i) \quad\text{(input gate)}$$
$$f_t = \sigma(x_t W_{xf} + h_{t-1} W_{hf} + b_f) \quad\text{(forget gate)}$$
$$g_t = \tanh(x_t W_{xg} + h_{t-1} W_{hg} + b_g) \quad\text{(cell candidate)}$$
$$o_t = \sigma(x_t W_{xo} + h_{t-1} W_{ho} + b_o) \quad\text{(output gate)}$$
$$c_t = f_t \odot c_{t-1} + i_t \odot g_t, \qquad h_t = o_t \odot \tanh(c_t)$$

We initialize **forget-gate bias to 1** so the cell defaults to remembering (standard practice).

**GRU** merges cell and hidden state with two gates (reset $r_t$, update $z_t$):

$$r_t = \sigma(\ldots), \quad z_t = \sigma(\ldots), \quad
n_t = \tanh(x_t W_{xn} + b_n + r_t \odot (h_{t-1} W_{hn}))$$
$$h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}$$

PyTorch applies the reset gate to the **hidden linear transform** $(h W_{hn})$, not to $h$ before
multiplication — our implementation matches that convention.

In [2]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))


def sigmoid_grad(y):
    return y * (1.0 - y)


def tanh(x):
    return np.tanh(x)


def tanh_grad(y):
    return 1.0 - y ** 2


class LSTMCell:
    def __init__(self, input_size, hidden_size, seed=0):
        self.input_size, self.hidden_size = input_size, hidden_size
        H = hidden_size
        rng = np.random.RandomState(seed)
        s = np.sqrt(2.0 / (input_size + hidden_size))
        self.W_xi = rng.randn(input_size, H) * s
        self.W_hi = rng.randn(H, H) * s
        self.b_i = np.zeros(H)
        self.W_xf = rng.randn(input_size, H) * s
        self.W_hf = rng.randn(H, H) * s
        self.b_f = np.ones(H)  # forget-gate bias init
        self.W_xg = rng.randn(input_size, H) * s
        self.W_hg = rng.randn(H, H) * s
        self.b_g = np.zeros(H)
        self.W_xo = rng.randn(input_size, H) * s
        self.W_ho = rng.randn(H, H) * s
        self.b_o = np.zeros(H)
        self.cache = None

    def forward(self, x, h, c):
        i = sigmoid(x @ self.W_xi + h @ self.W_hi + self.b_i)
        f = sigmoid(x @ self.W_xf + h @ self.W_hf + self.b_f)
        g = tanh(x @ self.W_xg + h @ self.W_hg + self.b_g)
        o = sigmoid(x @ self.W_xo + h @ self.W_ho + self.b_o)
        c_new = f * c + i * g
        tc = tanh(c_new)
        h_new = o * tc
        self.cache = (x, h, c, i, f, g, o, c_new, tc)
        return h_new, c_new

    def backward(self, dh, dc_next):
        x, h, c, i, f, g, o, c_new, tc = self.cache
        do = dh * tc
        dc = dc_next + dh * o * tanh_grad(tc)
        df, di, dg = dc * c, dc * g, dc * i
        dc_prev = dc * f
        di_in = di * sigmoid_grad(i)
        df_in = df * sigmoid_grad(f)
        dg_in = dg * tanh_grad(g)
        do_in = do * sigmoid_grad(o)
        dx = di_in @ self.W_xi.T + df_in @ self.W_xf.T + dg_in @ self.W_xg.T + do_in @ self.W_xo.T
        dh_prev = di_in @ self.W_hi.T + df_in @ self.W_hf.T + dg_in @ self.W_hg.T + do_in @ self.W_ho.T
        grads = dict(
            W_xi=x.T @ di_in, W_hi=h.T @ di_in, b_i=di_in.sum(0),
            W_xf=x.T @ df_in, W_hf=h.T @ df_in, b_f=df_in.sum(0),
            W_xg=x.T @ dg_in, W_hg=h.T @ dg_in, b_g=dg_in.sum(0),
            W_xo=x.T @ do_in, W_ho=h.T @ do_in, b_o=do_in.sum(0),
        )
        return dx, dh_prev, dc_prev, grads


class LSTM:
    def __init__(self, input_size, hidden_size, seed=0):
        self.input_size, self.hidden_size = input_size, hidden_size
        self.cell = LSTMCell(input_size, hidden_size, seed=seed)
        self.seq_cache = []

    def forward(self, x):
        N, T, D = x.shape
        H = self.hidden_size
        hs = np.zeros((N, T, H), dtype=x.dtype)
        h = np.zeros((N, H), dtype=x.dtype)
        c = np.zeros((N, H), dtype=x.dtype)
        self.seq_cache = []
        for t in range(T):
            h, c = self.cell.forward(x[:, t], h, c)
            self.seq_cache.append(self.cell.cache)
            hs[:, t] = h
        return hs

    def backward(self, dhs):
        N, T, _ = dhs.shape
        dx = np.zeros((N, T, self.input_size), dtype=dhs.dtype)
        dh = np.zeros((N, self.hidden_size), dtype=dhs.dtype)
        dc = np.zeros((N, self.hidden_size), dtype=dhs.dtype)
        keys = ['W_xi','W_hi','b_i','W_xf','W_hf','b_f','W_xg','W_hg','b_g','W_xo','W_ho','b_o']
        acc = {k: np.zeros_like(getattr(self.cell, k)) for k in keys}
        for t in reversed(range(T)):
            self.cell.cache = self.seq_cache[t]
            dx_t, dh, dc, grads = self.cell.backward(dhs[:, t] + dh, dc)
            dx[:, t] = dx_t
            for k, v in grads.items():
                acc[k] += v
        return dx, acc


class GRUCell:
    def __init__(self, input_size, hidden_size, seed=0):
        self.input_size, self.hidden_size = input_size, hidden_size
        H = hidden_size
        rng = np.random.RandomState(seed)
        s = np.sqrt(2.0 / (input_size + hidden_size))
        self.W_xr = rng.randn(input_size, H) * s
        self.W_hr = rng.randn(H, H) * s
        self.b_r = np.zeros(H)
        self.W_xz = rng.randn(input_size, H) * s
        self.W_hz = rng.randn(H, H) * s
        self.b_z = np.ones(H)  # update-gate bias init (carry memory by default)
        self.W_xn = rng.randn(input_size, H) * s
        self.W_hn = rng.randn(H, H) * s
        self.b_n = np.zeros(H)
        self.cache = None

    def forward(self, x, h):
        r = sigmoid(x @ self.W_xr + h @ self.W_hr + self.b_r)
        z = sigmoid(x @ self.W_xz + h @ self.W_hz + self.b_z)
        hn_pre = h @ self.W_hn
        n = tanh(x @ self.W_xn + self.b_n + r * hn_pre)
        h_new = (1.0 - z) * n + z * h
        self.cache = (x, h, r, z, n, hn_pre)
        return h_new

    def backward(self, dh):
        x, h, r, z, n, hn_pre = self.cache
        dn = dh * (1.0 - z)
        dz = dh * (h - n)
        dh_prev = dh * z
        dn_in = dn * tanh_grad(n)
        dhn_pre = dn_in * r
        dr = dn_in * hn_pre
        dz_in = dz * sigmoid_grad(z)
        dr_in = dr * sigmoid_grad(r)
        dx = dn_in @ self.W_xn.T + dz_in @ self.W_xz.T + dr_in @ self.W_xr.T
        dh_prev += dhn_pre @ self.W_hn.T + dz_in @ self.W_hz.T + dr_in @ self.W_hr.T
        grads = dict(
            W_xn=x.T @ dn_in, W_hn=h.T @ dhn_pre, b_n=dn_in.sum(0),
            W_xz=x.T @ dz_in, W_hz=h.T @ dz_in, b_z=dz_in.sum(0),
            W_xr=x.T @ dr_in, W_hr=h.T @ dr_in, b_r=dr_in.sum(0),
        )
        return dx, dh_prev, grads


class GRU:
    def __init__(self, input_size, hidden_size, seed=0):
        self.input_size, self.hidden_size = input_size, hidden_size
        self.cell = GRUCell(input_size, hidden_size, seed=seed)
        self.seq_cache = []

    def forward(self, x):
        N, T, D = x.shape
        H = self.hidden_size
        hs = np.zeros((N, T, H), dtype=x.dtype)
        h = np.zeros((N, H), dtype=x.dtype)
        self.seq_cache = []
        for t in range(T):
            h = self.cell.forward(x[:, t], h)
            self.seq_cache.append(self.cell.cache)
            hs[:, t] = h
        return hs

    def backward(self, dhs):
        N, T, _ = dhs.shape
        dx = np.zeros((N, T, self.input_size), dtype=dhs.dtype)
        dh = np.zeros((N, self.hidden_size), dtype=dhs.dtype)
        keys = ['W_xr','W_hr','b_r','W_xz','W_hz','b_z','W_xn','W_hn','b_n']
        acc = {k: np.zeros_like(getattr(self.cell, k)) for k in keys}
        for t in reversed(range(T)):
            self.cell.cache = self.seq_cache[t]
            dx_t, dh, grads = self.cell.backward(dhs[:, t] + dh)
            dx[:, t] = dx_t
            for k, v in grads.items():
                acc[k] += v
        return dx, acc

print('LSTM and GRU defined.')

LSTM and GRU defined.


## 2. Validation Against PyTorch

Gate weight matrices are stacked in PyTorch order (LSTM: i,f,g,o; GRU: r,z,n).

In [3]:
def stack_lstm_weights(cell):
    W_ih = np.concatenate([cell.W_xi.T, cell.W_xf.T, cell.W_xg.T, cell.W_xo.T], axis=0)
    W_hh = np.concatenate([cell.W_hi.T, cell.W_hf.T, cell.W_hg.T, cell.W_ho.T], axis=0)
    b = np.concatenate([cell.b_i, cell.b_f, cell.b_g, cell.b_o])
    return W_ih, W_hh, b


def stack_gru_weights(cell):
    W_ih = np.concatenate([cell.W_xr.T, cell.W_xz.T, cell.W_xn.T], axis=0)
    W_hh = np.concatenate([cell.W_hr.T, cell.W_hz.T, cell.W_hn.T], axis=0)
    b = np.concatenate([cell.b_r, cell.b_z, cell.b_n])
    return W_ih, W_hh, b


rng42 = np.random.RandomState(42)
N, T, D, H = 3, 5, 4, 6
x_np = rng42.randn(N, T, D).astype(np.float64)

# --- LSTM ---
lstm = LSTM(D, H, seed=1)
c = lstm.cell
for name in ['W_xi','W_hi','W_xf','W_hf','W_xg','W_hg','W_xo','W_ho']:
    setattr(c, name, rng42.randn(*getattr(c, name).shape).astype(np.float64) * 0.2)
for bn in ['b_i','b_g','b_o']:
    setattr(c, bn, rng42.randn(H).astype(np.float64) * 0.01)
c.b_f = np.ones(H)
hs = lstm.forward(x_np)
dhs = rng42.randn(N, T, H).astype(np.float64)
dx, lgrads = lstm.backward(dhs)
x_t = torch.tensor(x_np, requires_grad=True, dtype=torch.float64)
lt = nn.LSTM(D, H, batch_first=True, num_layers=1).double()
W_ih, W_hh, b = stack_lstm_weights(c)
with torch.no_grad():
    lt.weight_ih_l0.copy_(torch.tensor(W_ih))
    lt.weight_hh_l0.copy_(torch.tensor(W_hh))
    lt.bias_ih_l0.copy_(torch.tensor(b))
    lt.bias_hh_l0.zero_()
out, _ = lt(x_t)
(out * torch.tensor(dhs)).sum().backward()
print('LSTM validation:')
print(f"  forward max diff: {np.abs(hs - out.detach().numpy()).max():.2e}")
print(f"  dx max diff:      {np.abs(dx - x_t.grad.numpy()).max():.2e}")

# --- GRU ---
gru = GRU(D, H, seed=1)
gc = gru.cell
for name in ['W_xr','W_hr','W_xz','W_hz','W_xn','W_hn']:
    setattr(gc, name, rng42.randn(*getattr(gc, name).shape).astype(np.float64) * 0.2)
for bn in ['b_r','b_n']:
    setattr(gc, bn, rng42.randn(H).astype(np.float64) * 0.01)
gc.b_z = np.ones(H)
ghs = gru.forward(x_np)
gdx, ggrads = gru.backward(dhs)
x_t2 = torch.tensor(x_np, requires_grad=True, dtype=torch.float64)
gt = nn.GRU(D, H, batch_first=True, num_layers=1).double()
W_ih, W_hh, b = stack_gru_weights(gc)
with torch.no_grad():
    gt.weight_ih_l0.copy_(torch.tensor(W_ih))
    gt.weight_hh_l0.copy_(torch.tensor(W_hh))
    gt.bias_ih_l0.copy_(torch.tensor(b))
    gt.bias_hh_l0.zero_()
gout, _ = gt(x_t2)
(gout * torch.tensor(dhs)).sum().backward()
print('\nGRU validation:')
print(f"  forward max diff: {np.abs(ghs - gout.detach().numpy()).max():.2e}")
print(f"  dx max diff:      {np.abs(gdx - x_t2.grad.numpy()).max():.2e}")

LSTM validation:
  forward max diff: 5.55e-17
  dx max diff:      5.55e-17

GRU validation:
  forward max diff: 9.02e-17
  dx max diff:      1.67e-16


## 3. Gradient Flow: Vanilla RNN vs LSTM

Same setup as Topic 10: impulse at $t=0$, loss on final hidden state only.
The LSTM's additive cell-state path ($c_t = f_t \odot c_{t-1} + \ldots$) lets gradients
flow without repeated $\tanh$ squashing through $W_{hh}$.

In [4]:
def tanh_grad_z(z):
    return 1.0 - np.tanh(z) ** 2


class VanillaRNN:
    def __init__(self, input_size, hidden_size, seed=0):
        self.input_size, self.hidden_size = input_size, hidden_size
        rng = np.random.RandomState(seed)
        self.W_xh = rng.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.W_hh = rng.randn(hidden_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        self.b = np.zeros(hidden_size)
        self.seq_cache = []

    def forward(self, x):
        N, T, D = x.shape
        H = self.hidden_size
        hs = np.zeros((N, T, H), dtype=x.dtype)
        h = np.zeros((N, H), dtype=x.dtype)
        self.seq_cache = []
        for t in range(T):
            z = x[:, t] @ self.W_xh + h @ self.W_hh + self.b
            h = tanh(z)
            self.seq_cache.append((x[:, t], h.copy(), z))
            hs[:, t] = h
        return hs


def grad_profile_vanilla(T=40, hidden=16, whh_scale=0.9):
    rnn = VanillaRNN(1, hidden, seed=0)
    rnn.W_hh = np.eye(hidden) * whh_scale
    rnn.W_xh *= 0.1
    x = np.zeros((1, T, 1), dtype=np.float64)
    x[0, 0, 0] = 1.0
    rnn.forward(x)
    dh = np.zeros((1, hidden))
    dhs = np.zeros((1, T, hidden))
    dhs[0, -1, 0] = 1.0
    norms = []
    for t in reversed(range(T)):
        x_t, h_prev, z = rnn.seq_cache[t]
        dh += dhs[0, t]
        norms.append(float(np.linalg.norm(dh)))
        dh = (dh * tanh_grad_z(z)) @ rnn.W_hh.T
    return list(reversed(norms))


def grad_profile_lstm(T=40, hidden=16):
    lstm = LSTM(1, hidden, seed=0)
    x = np.zeros((1, T, 1), dtype=np.float64)
    x[0, 0, 0] = 1.0
    lstm.forward(x)
    dh = np.zeros((1, hidden))
    dc = np.zeros((1, hidden))
    dhs = np.zeros((1, T, hidden))
    dhs[0, -1, 0] = 1.0
    norms = []
    for t in reversed(range(T)):
        lstm.cell.cache = lstm.seq_cache[t]
        dx_t, dh, dc, _ = lstm.cell.backward(dhs[0, t] + dh, dc)
        norms.append(float(np.linalg.norm(dh + dc)))
    return list(reversed(norms))


T = 40
nv = grad_profile_vanilla(T)
nl = grad_profile_lstm(T)
print(f"Vanilla RNN: ratio ||dh_0||/||dh_{T-1}|| = {nv[0]/nv[-1]:.4e}")
print(f"LSTM:        ratio ||dh_0||/||dh_{T-1}|| = {nl[0]/nl[-1]:.4e}")
print(f"Early-step gradient: vanilla={nv[0]:.4e}, LSTM={nl[0]:.4e} ({nl[0]/nv[0]:.1f}x larger)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(range(T), nv, '-o', ms=3, label='Vanilla RNN (W_hh=0.9I)')
ax.semilogy(range(T), nl, '-o', ms=3, label='LSTM (forget bias=1)')
ax.set_xlabel('timestep t')
ax.set_ylabel('gradient norm (log scale)')
ax.set_title('BPTT gradient magnitude (loss at final step)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('lstm_grad_flow.png', dpi=100, bbox_inches='tight')
plt.show()

Vanilla RNN: ratio ||dh_0||/||dh_39|| = 1.2969e-02
LSTM:        ratio ||dh_0||/||dh_39|| = 3.3355e-01
Early-step gradient: vanilla=1.2969e-02, LSTM=9.9737e-02 (7.7x larger)


C:\Users\PM_IntellicaBD\AppData\Local\Temp\ipykernel_21364\627700903.py:80: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Parameter Count: LSTM vs GRU vs Vanilla RNN

For hidden size $H$ and input size $D$:

In [5]:
D, H = 1, 32
vanilla = H * D + H * H + H
lstm_p = 4 * (H * D + H * H + H)
gru_p = 3 * (H * D + H * H + H)
print(f"Vanilla RNN params (D={D}, H={H}): {vanilla}")
print(f"LSTM params:                      {lstm_p}  ({lstm_p/vanilla:.1f}x vanilla)")
print(f"GRU params:                       {gru_p}  ({gru_p/vanilla:.1f}x vanilla)")

Vanilla RNN params (D=1, H=32): 1088
LSTM params:                      4352  (4.0x vanilla)
GRU params:                       3264  (3.0x vanilla)


## 5. Delayed Recall: Where Vanilla RNNs Fail (Topic 10)

Label encoded at $t=0$, noise for $t=1\ldots T-1$. Topic 10 showed vanilla RNN mean
accuracy $\approx 0.47$ at $T=50$. We re-run with LSTM and GRU (hidden=32, 3 seeds).

In [6]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


class SeqClassifier:
    def __init__(self, rnn_type, input_size, hidden_size, seed=0):
        self.rnn_type = rnn_type
        if rnn_type == 'vanilla':
            self.core = VanillaRNN(input_size, hidden_size, seed=seed)
        elif rnn_type == 'lstm':
            self.core = LSTM(input_size, hidden_size, seed=seed)
        else:
            self.core = GRU(input_size, hidden_size, seed=seed)
        rng = np.random.RandomState(seed + 1)
        self.W_hy = rng.randn(hidden_size, 2) * np.sqrt(2.0 / hidden_size)
        self.b_y = np.zeros(2)

    def forward(self, x):
        hs = self.core.forward(x)
        return hs, hs[:, -1] @ self.W_hy + self.b_y

    def step(self, x, y, lr=0.05):
        hs, logits = self.forward(x)
        probs = softmax(logits)
        n = len(y)
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))
        dlogits = probs.copy()
        dlogits[np.arange(n), y] -= 1
        dlogits /= n
        self.W_hy -= lr * (hs[:, -1].T @ dlogits)
        self.b_y -= lr * dlogits.sum(0)
        dhs = np.zeros_like(hs)
        dhs[:, -1] = dlogits @ self.W_hy.T
        if self.rnn_type == 'vanilla':
            N, T, D = x.shape
            dx = np.zeros_like(x)
            dW_xh = np.zeros_like(self.core.W_xh)
            dW_hh = np.zeros_like(self.core.W_hh)
            db = np.zeros_like(self.core.b)
            dh = np.zeros((N, self.core.hidden_size))
            for t in reversed(range(T)):
                x_t, h_prev, z = self.core.seq_cache[t]
                dh += dhs[:, t]
                dz = dh * tanh_grad_z(z)
                dW_xh += x_t.T @ dz
                dW_hh += h_prev.T @ dz
                db += dz.sum(0)
                dh = dz @ self.core.W_hh.T
            self.core.W_xh -= lr * dW_xh
            self.core.W_hh -= lr * dW_hh
            self.core.b -= lr * db
        else:
            dx, acc = self.core.backward(dhs)
            cell = self.core.cell
            for k, v in acc.items():
                setattr(cell, k, getattr(cell, k) - lr * v)
        return loss

    def predict(self, x):
        return self.forward(x)[1].argmax(axis=1)


def make_delay_data(n=800, T=50, seed=0):
    rng = np.random.RandomState(seed)
    X = rng.randn(n, T, 1) * 0.5
    y = (rng.rand(n) > 0.5).astype(int)
    X[:, 0, 0] = np.where(y, 1.0, -1.0)
    idx = rng.permutation(n)
    split = int(0.75 * n)
    tr, te = idx[:split], idx[split:]
    return X[tr], y[tr], X[te], y[te]


def train_delay(rnn_type, T, seed, epochs=100, lr=0.05, hidden=32):
    Xtr, ytr, Xte, yte = make_delay_data(T=T, seed=seed)
    model = SeqClassifier(rnn_type, 1, hidden, seed=seed)
    rng = np.random.RandomState(0)
    for _ in range(epochs):
        idx = rng.permutation(len(ytr))
        for i in range(0, len(idx), 32):
            b = idx[i:i + 32]
            model.step(Xtr[b], ytr[b], lr=lr)
    return (model.predict(Xte) == yte).mean()


print(f"{'Model':>8} {'seed=0':>8} {'seed=1':>8} {'seed=2':>8} {'mean':>8}")
for rnn_type in ['vanilla', 'lstm', 'gru']:
    accs = [train_delay(rnn_type, T=50, seed=s, epochs=100, lr=0.05, hidden=32) for s in range(3)]
    print(f"{rnn_type:>8} {accs[0]:>8.3f} {accs[1]:>8.3f} {accs[2]:>8.3f} {np.mean(accs):>8.3f}")

   Model   seed=0   seed=1   seed=2     mean


 vanilla    0.560    0.450    0.525    0.512


    lstm    1.000    0.995    0.995    0.997


     gru    0.595    0.995    0.995    0.862


## 6. Longer Sequences ($T=100$)

At $T=100$, training is harder for all models. LSTM retains an edge on some seeds
while vanilla and GRU hover near chance.

In [7]:
print(f"{'Model':>8} {'seed=0':>8} {'seed=1':>8} {'seed=2':>8} {'mean':>8}")
for rnn_type in ['vanilla', 'lstm', 'gru']:
    accs = [train_delay(rnn_type, T=100, seed=s, epochs=200, lr=0.01, hidden=32) for s in range(3)]
    print(f"{rnn_type:>8} {accs[0]:>8.3f} {accs[1]:>8.3f} {accs[2]:>8.3f} {np.mean(accs):>8.3f}")

   Model   seed=0   seed=1   seed=2     mean


 vanilla    0.425    0.455    0.525    0.468


    lstm    0.475    1.000    0.490    0.655


     gru    0.480    0.505    0.475    0.487
